# Paid Search — ML Model Training
Optimise conversion prediction for **PPC / Paid Search** campaigns.

**Connected to `ml_utils.py` v3 engine** — trains on the real `digital_marketing_campaign_dataset.csv` (8,000 records) and integrates with the global Voting Ensemble.

## 1. Imports & Setup
Load all libraries and locate `ml_utils.py` automatically.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# ── ml_utils path setup ─────────────────────────────────────────────────────
# Add project root so ml_utils imports cleanly
_HERE = os.path.abspath(os.path.dirname("__file__") if "__file__" in dir() else os.getcwd())
for _candidate in [_HERE, os.path.join(_HERE, '..'), os.path.join(_HERE, '../..')]:
    if os.path.exists(os.path.join(_candidate, 'ml_utils.py')):
        sys.path.insert(0, _candidate)
        break

from ml_utils import ( # type: ignore # type: ignore # type: ignore
    _get_engine, _train_engine, _engineer_features,
    CONV_FEATURES_V3, CHANNEL_MAP_CSV, CHANNEL_MAP_XLSX,
    CHANNELS, CHANNEL_NAMES, STRATEGY_CONFIG,
    get_model_metrics, predict_channel_roi, get_channel_model_info
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                               ExtraTreesClassifier, VotingClassifier)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                              precision_recall_curve, average_precision_score)
from sklearn.inspection import permutation_importance
import pickle

print("✓ All libraries loaded")
print(f"  ml_utils found at: {sys.path[0]}")


ModuleNotFoundError: No module named 'ml_utils'

## 2. Load Datasets
Load both CSVs directly from disk — no random data generation.

In [ ]:
# ── Locate datasets ───────────────────────────────────────────────────────────
_ROOT = sys.path[0]
CSV_PATH  = None
XLSX_PATH = None
for _base in [_ROOT, os.getcwd()]:
    for _name in ['digital_marketing_campaign_dataset.csv']:
        _p = os.path.join(_base, _name)
        if os.path.exists(_p): CSV_PATH = _p
    for _name in ['marketing_campaign_dataset.xlsx']:
        _p = os.path.join(_base, _name)
        if os.path.exists(_p): XLSX_PATH = _p
    for _sub in ['data']:
        for _name in ['digital_marketing_campaign_dataset.csv']:
            _p = os.path.join(_base, _sub, _name)
            if os.path.exists(_p): CSV_PATH = _p
        for _name in ['marketing_campaign_dataset.xlsx']:
            _p = os.path.join(_base, _sub, _name)
            if os.path.exists(_p): XLSX_PATH = _p

print(f"CSV  dataset : {CSV_PATH}")
print(f"XLSX dataset : {XLSX_PATH}")

df_csv  = pd.read_csv(CSV_PATH)
df_xlsx = pd.read_excel(XLSX_PATH)

print(f"\nCSV  shape : {df_csv.shape}  | columns: {df_csv.columns.tolist()}")
print(f"XLSX shape : {df_xlsx.shape} | columns: {df_xlsx.columns.tolist()[:10]}...")
print(f"\nCSV dtypes:\n{df_csv.dtypes}")
print(f"\nMissing values:\n{df_csv.isnull().sum()[df_csv.isnull().sum()>0]}")


## 3. Exploratory Data Analysis (Full Dataset)
Understand the overall dataset before drilling into channel-specific data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Dataset EDA — Digital Marketing Campaign (n=8,000)", fontsize=14, fontweight='bold')

# Conversion rate distribution
df_csv['Conversion'].value_counts().plot.bar(ax=axes[0,0], color=['#e74c3c','#2ecc71'])
axes[0,0].set_title('Conversion Distribution'); axes[0,0].set_xlabel('')

# Channel distribution
df_csv['CampaignChannel'].value_counts().plot.barh(ax=axes[0,1], color=sns.color_palette("husl",5))
axes[0,1].set_title('Records per Channel')

# AdSpend distribution
axes[0,2].hist(df_csv['AdSpend'], bins=40, color='#3498db', edgecolor='white')
axes[0,2].set_title('AdSpend Distribution'); axes[0,2].set_xlabel('AdSpend (USD)')

# CTR vs ConversionRate
axes[1,0].scatter(df_csv['ClickThroughRate'], df_csv['ConversionRate'],
                  c=df_csv['Conversion'], cmap='RdYlGn', alpha=0.4, s=10)
axes[1,0].set_xlabel('CTR'); axes[1,0].set_ylabel('CVR'); axes[1,0].set_title('CTR vs CVR (colored by Conversion)')

# Age distribution
df_csv['Age'].hist(ax=axes[1,1], bins=30, color='#9b59b6', edgecolor='white')
axes[1,1].set_title('Age Distribution')

# Correlation heatmap
num_cols = ['AdSpend','ClickThroughRate','ConversionRate','WebsiteVisits',
            'PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks',
            'PreviousPurchases','LoyaltyPoints','Conversion']
corr = df_csv[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=axes[1,2], cmap='coolwarm', center=0,
            annot=False, fmt='.2f', linewidths=0.5)
axes[1,2].set_title('Feature Correlations')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓ EDA complete")


## 4. Feature Engineering (via `ml_utils._engineer_features`)
All 27 features are computed using the exact same pipeline as the global model — no divergence.

In [ ]:
# ── Feature Engineering via ml_utils ─────────────────────────────────────────
le_ch = LabelEncoder().fit(df_csv['CampaignChannel'])
le_ct = LabelEncoder().fit(df_csv['CampaignType'])

df_feat = _engineer_features(df_csv, le_ch, le_ct)
df_feat['channel_key'] = df_csv['CampaignChannel'].map(CHANNEL_MAP_CSV)

print("✓ Feature engineering complete")
print(f"  Total features available : {len(CONV_FEATURES_V3)}")
print(f"  Feature list (CONV_FEATURES_V3):")
for i, f in enumerate(CONV_FEATURES_V3, 1):
    print(f"    {i:2}. {f}")

X_full = df_feat[CONV_FEATURES_V3].replace([np.inf,-np.inf], np.nan).fillna(0)
y_full = df_feat['Conversion']
print(f"\nX shape : {X_full.shape} | y shape : {y_full.shape}")
print(f"Class balance — Conversion=1: {y_full.mean():.1%}  Conversion=0: {(1-y_full.mean()):.1%}")


## 5. Paid Search Channel Filter
Isolate records for this channel only.

In [ ]:
# ── Filter to Paid Search records ─────────────────────────────────────────
CHANNEL_KEY = 'paid_search'

df_ch = df_feat[df_feat['channel_key'] == CHANNEL_KEY].copy()
X_ch  = df_ch[CONV_FEATURES_V3].replace([np.inf,-np.inf], np.nan).fillna(0)
y_ch  = df_ch['Conversion']

print(f"Channel        : Paid Search")
print(f"Records        : {len(df_ch)}")
print(f"Conversion rate: {y_ch.mean():.1%}")
print(f"AdSpend (mean) : ${df_ch['AdSpend'].mean():,.0f}")
print(f"CTR (mean)     : {df_ch['ClickThroughRate'].mean():.4f}")
print(f"CVR (mean)     : {df_ch['ConversionRate'].mean():.4f}")


## 6. Paid Search Channel EDA
Visualise channel-specific distributions.

In [ ]:
# ── Paid Search — Channel EDA ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Paid Search — Exploratory Analysis", fontsize=13, fontweight='bold')

# Conversion split
df_ch['Conversion'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('Conversion (0/1)'); axes[0].set_xticklabels(['No','Yes'], rotation=0)

# CTR distribution
df_ch['ClickThroughRate'].hist(ax=axes[1], bins=30, color='#3498db', edgecolor='white')
axes[1].set_title('CTR Distribution'); axes[1].set_xlabel('Click-Through Rate')

# AdSpend vs Conversion
for v, col in zip([0,1],['#e74c3c','#2ecc71']):
    axes[2].hist(df_ch[df_ch['Conversion']==v]['AdSpend'], bins=25, alpha=0.6,
                 color=col, label=f'Conv={v}', edgecolor='white')
axes[2].set_title('AdSpend by Conversion'); axes[2].set_xlabel('AdSpend')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'paid_search_eda.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Train Paid Search Channel Ensemble
Same architecture as the global model: GBM + RandomForest + ExtraTrees with Isotonic Calibration.

In [ ]:
# ── Train Channel-Specific Ensemble ──────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
import time

X_tr, X_te, y_tr, y_te = train_test_split(
    X_ch, y_ch, test_size=0.2, random_state=42, stratify=y_ch
)

print(f"Train : {len(X_tr)} | Test : {len(X_te)}")
print("Training Paid Search ensemble model...")
t0 = time.time()

gbm = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=10, random_state=42,
)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=8,
    max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1,
)
et = ExtraTreesClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=8,
    max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1,
)

ensemble = VotingClassifier(
    estimators=[('gbm', gbm), ('rf', rf), ('et', et)], voting='soft'
)
model_ch = CalibratedClassifierCV(ensemble, method='isotonic', cv=3)
model_ch.fit(X_tr, y_tr)

elapsed = time.time() - t0
holdout_auc = roc_auc_score(y_te, model_ch.predict_proba(X_te)[:,1])
ap_score    = average_precision_score(y_te, model_ch.predict_proba(X_te)[:,1])

print(f"\n✓ Training done in {elapsed:.1f}s")
print(f"  Holdout AUC    : {holdout_auc:.4f}")
print(f"  Avg Precision  : {ap_score:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_te, model_ch.predict(X_te), target_names=['No Convert','Convert']))


## 8. Cross-Validation (5-Fold Stratified)
Robust AUC estimation with no data leakage.

In [ ]:
# ── 5-Fold Cross-Validation ───────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Running 5-fold stratified cross-validation...")

cv_pipeline = CalibratedClassifierCV(
    VotingClassifier(estimators=[
        ('gbm', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                           max_depth=4, subsample=0.8, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=200, max_depth=8,
                                       class_weight='balanced', random_state=42, n_jobs=-1)),
        ('et',  ExtraTreesClassifier(n_estimators=200, max_depth=8,
                                     class_weight='balanced', random_state=42, n_jobs=-1)),
    ], voting='soft'),
    method='isotonic', cv=3
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_pipeline, X_ch, y_ch, cv=skf,
                             scoring='roc_auc', n_jobs=-1)

print(f"\n5-Fold CV AUC scores : {cv_scores}")
print(f"Mean ± Std           : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Min / Max            : {cv_scores.min():.4f} / {cv_scores.max():.4f}")


## 9. Feature Importance & Model Evaluation
Visualise what drives conversions for this channel.

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
try:
    gbm_sub = model_ch.calibrated_classifiers_[0].estimator.estimators_[0][1]
    importances = gbm_sub.feature_importances_
except Exception:
    importances = np.ones(len(CONV_FEATURES_V3)) / len(CONV_FEATURES_V3)

feat_imp = pd.Series(importances, index=CONV_FEATURES_V3).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Paid Search — Model Evaluation", fontsize=13, fontweight='bold')

# Feature importance
feat_imp.tail(15).plot.barh(ax=axes[0], color='#3498db')
axes[0].set_title('Top 15 Feature Importances (GBM)'); axes[0].set_xlabel('Importance')

# Precision-Recall Curve
probs_te = model_ch.predict_proba(X_te)[:,1]
precision, recall, _ = precision_recall_curve(y_te, probs_te)
axes[1].plot(recall, precision, color='#e74c3c', lw=2,
             label=f'AP = {ap_score:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'paid_search_model_eval.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 10 features for Paid Search:")
for feat, imp in feat_imp.tail(10).items():
    print(f"  {feat:<35} {imp:.4f}")


NameError: name 'np' is not defined

## 10. Connect to `ml_utils` Global Engine
Verify this notebook's results against the global engine and pull channel profiles.

In [2]:
# ── Connect to ml_utils — pull global engine metrics ─────────────────────────
engine = _get_engine()

ch_profile = engine['channel_profiles'].get('paid_search', {})
global_metrics = engine['metrics']['conversion_model']

print("=" * 60)
print(f"  ml_utils ENGINE — Paid Search")
print("=" * 60)
print(f"  Global model type   : {global_metrics.get('model_type')}")
print(f"  Global holdout AUC  : {global_metrics.get('auc')}")
print(f"  Global 5-fold AUC   : {global_metrics.get('cv_auc_mean')} ± {global_metrics.get('cv_auc_std')}")
print(f"  Training samples    : {global_metrics.get('training_samples')}")
print(f"  Features used       : {global_metrics.get('feature_count')}")
print()
print(f"  Channel Profile — Paid Search")
print(f"  Conv probability    : {ch_profile.get('conv_prob')}")
print(f"  Conv CI (95%)       : {ch_profile.get('conv_ci')}")
print(f"  CTR                 : {ch_profile.get('ctr')}")
print(f"  CVR                 : {ch_profile.get('cvr')}")
print(f"  Engagement score    : {ch_profile.get('engagement')}")
print(f"  Cost per acquisition: ₹{ch_profile.get('cost_per_acq'):,.0f}")
print(f"  Cost efficiency     : {ch_profile.get('cost_efficiency')}")
print(f"  Avg spend (INR)     : ₹{ch_profile.get('avg_spend'):,.0f}")

# Compare channel-specific model vs global
print()
print(f"  Channel-specific AUC (this notebook) : {holdout_auc:.4f}")
print(f"  Global ensemble AUC (ml_utils)        : {global_metrics.get('auc')}")


NameError: name '_get_engine' is not defined

## 11. Live ROI Prediction via `ml_utils` API
Call `predict_channel_roi()` as the backend does.

In [ ]:
# ── ROI Prediction via ml_utils API ──────────────────────────────────────────
print("Testing predict_channel_roi() from ml_utils...")
print()

test_scenarios = [
    {"spend": 500_000,    "label": "Small budget (₹5L)"},
    {"spend": 2_000_000,  "label": "Medium budget (₹20L)"},
    {"spend": 10_000_000, "label": "Large budget (₹1Cr)"},
]

for scenario in test_scenarios:
    result = predict_channel_roi('paid_search', 'notebook_user',
                                  {'spend': scenario['spend']})
    print(f"  {scenario['label']}:")
    print(f"    Predicted ROI     : {result.get('predicted_roi', 'N/A'):.1f}%")
    print(f"    Conv Probability  : {result.get('conv_probability', 'N/A'):.1f}%")
    print(f"    CTR               : {result.get('ctr', 'N/A'):.2f}%")
    print(f"    CVR               : {result.get('cvr', 'N/A'):.2f}%")
    print(f"    Model Confidence  : {result.get('model_confidence', 'N/A'):.4f}")
    print()


## 12. Save Model & Metadata
Persist the channel model alongside the global engine artifacts.

In [1]:
# ── Save Channel Model + Metadata ────────────────────────────────────────────
MODEL_DIR = os.path.join(sys.path[0], 'ml_models')
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, f'paid_search_channel_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_ch, f)

metadata = {
    'channel':          'paid_search',
    'channel_name':     'Paid Search',
    'model_type':       'Ensemble(GBM+RF+ET)+IsotonicCalibration',
    'feature_count':    len(CONV_FEATURES_V3),
    'features':         CONV_FEATURES_V3,
    'training_samples': len(X_ch),
    'holdout_auc':      round(holdout_auc, 4),
    'avg_precision':    round(ap_score, 4),
    'cv_auc_mean':      round(float(cv_scores.mean()), 4),
    'cv_auc_std':       round(float(cv_scores.std()), 4),
    'ml_utils_profile': engine['channel_profiles'].get('paid_search', {}),
    'trained_at':       datetime.utcnow().isoformat(),
    'notebook':         'paid_search_model.ipynb',
}

meta_path = os.path.join(MODEL_DIR, f'paid_search_channel_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("✓ Paid Search channel model saved")
print(f"  Model     : {model_path}")
print(f"  Metadata  : {meta_path}")
print(f"\nFinal Metrics Summary:")
print(f"  Holdout AUC  : {holdout_auc:.4f}")
print(f"  Avg Precision: {ap_score:.4f}")
print(f"  CV AUC       : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Samples      : {len(X_ch)}")
print(f"  Features     : {len(CONV_FEATURES_V3)}")


NameError: name 'os' is not defined

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Load dataset
df = pd.read_csv("../../marketingapp/data/digital_marketing_campaign_dataset.csv")

# Show dataset
print(df.head())

# Convert categorical columns to numeric
df = pd.get_dummies(df)

# Target column
target_column = "Conversion"

X = df.drop(columns=[target_column])
y = df[target_column]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nModel Performance")
print("---------------------")
print("R2 Score:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

   CustomerID  Age  Gender  Income CampaignChannel CampaignType      AdSpend  \
0        8000   56  Female  136912    Social Media    Awareness  6497.870068   
1        8001   69    Male   41760           Email    Retention  3898.668606   
2        8002   46  Female   88456             PPC    Awareness  1546.429596   
3        8003   32  Female   44085             PPC   Conversion   539.525936   
4        8004   60  Female   83964             PPC   Conversion  1678.043573   

   ClickThroughRate  ConversionRate  WebsiteVisits  PagesPerVisit  TimeOnSite  \
0          0.043919        0.088031              0       2.399017    7.396803   
1          0.155725        0.182725             42       2.917138    5.352549   
2          0.277490        0.076423              2       8.223619   13.794901   
3          0.137611        0.088004             47       4.540939   14.688363   
4          0.252851        0.109940              0       2.046847   13.993370   

   SocialShares  EmailOpens  Ema

## Summary
### Paid Search Key Insights
- PPC has the highest direct intent signal — `ClickThroughRate` and `AdSpend` are top predictors.
- `spend_per_visit` and `email_engagement` cross-channel signals still matter.
- Isotonic calibration corrects for overconfidence in high-CTR segments.

**How this notebook connects to the backend:**
- `_get_engine()` → loads/trains the global ensemble
- `_engineer_features()` → identical feature pipeline
- `predict_channel_roi()` → used by `views.py` at inference time
- `get_channel_model_info()` → surfaces metrics to the dashboard
- Channel model saved to `ml_models/paid_search_channel_model.pkl`